# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata  # 'metadata' is a Croissant Dataset metadata object

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets and their IDs, then inspect fields within each record set.

All references to record sets, fields, and columns in this notebook will use their `@id` according to the Croissant specification.

In [ ]:
# Display available record sets and fields, referencing by `@id`
if hasattr(metadata, 'record_sets'):
    print('Available record sets:')
    record_sets = metadata.record_sets
    record_set_ids = []
    for record_set in record_sets:
        print(f"  - @id: {record_set.id} | name: {record_set.name}")
        record_set_ids.append(record_set.id)
        print("    Fields and their @id in record set:")
        for field in record_set.fields:
            print(f"      * @id: {field.id} | name: {field.name} | data_type: {field.data_type}")
else:
    print('No record sets found in the metadata.')

## 3. Data Extraction
Load data from all available record sets into DataFrames for analysis. Use the record set and field `@id`s from the overview. Display a sample of the data.

In [ ]:
# Load the data from all record sets
dataframes = {}

# The variable record_set_ids was built above when displaying overview
for record_set_id in record_set_ids:
    print(f"Loading records for record set: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Columns (@id): {df.columns.tolist()}")
    print(df.head(), '\n')

# For demonstration, pick the first record set for further steps
main_record_set_id = record_set_ids[0] if record_set_ids else None

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

This section demonstrates basic exploratory analysis using field `@id`s. Adjust the field `@id` and logic to fit the real schema if different.

In [ ]:
import numpy as np
# Example: Suppose the GAD-7 total score field `@id` is 'gad7_total_score'
# and a grouping demographic field `@id` is 'gender' (these should be exact @id values)

# SET THE FIELD IDS for analysis (update below if actual @ids differ)
gad7_field_id = None
gender_field_id = None
if main_record_set_id is not None:
    df = dataframes[main_record_set_id]
    # Try to guess the best column for GAD-7 and gender based on name substrings
    for col in df.columns:
        if 'gad' in col.lower() and 'score' in col.lower():
            gad7_field_id = col
        if 'gender' in col.lower():
            gender_field_id = col
    if gad7_field_id:
        print(f"Using GAD-7 field @id for analysis: {gad7_field_id}")
    else:
        print("No suitable GAD-7 total score field found in the columns.")
    if gender_field_id:
        print(f"Using gender field @id for grouping: {gender_field_id}")
    else:
        print("No suitable gender field found in the columns.")

    if gad7_field_id:
        # Filter: GAD-7 > 10 (typically threshold for moderate anxiety)
        # Try numeric casting in case the column is not float already
        df[gad7_field_id] = pd.to_numeric(df[gad7_field_id], errors='coerce')
        threshold = 10
        filtered_df = df[df[gad7_field_id] > threshold].copy()
        print(f"Filtered records with {gad7_field_id} > {threshold}:")
        print(filtered_df[[gad7_field_id]].head())

        # Normalization
        filtered_df[f"{gad7_field_id}_normalized"] = (
            (filtered_df[gad7_field_id] - filtered_df[gad7_field_id].mean()) / filtered_df[gad7_field_id].std()
        )
        print(f"Normalized {gad7_field_id} for filtered records:")
        print(filtered_df[[gad7_field_id, f"{gad7_field_id}_normalized"]].head())

        # Group by Gender (if exists)
        if gender_field_id in filtered_df.columns:
            grouped = filtered_df.groupby(gender_field_id)[gad7_field_id].agg(['mean', 'count'])
            print(f"Grouped data by gender ({gender_field_id}):\n", grouped.head())
else:
    print('No main record set available for EDA.')

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Below, we plot the distribution of GAD-7 scores and mean GAD-7 score by gender if present. You can extend this to PSQ or PHQ-9 fields by using their `@id`s.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_id is not None and gad7_field_id and gad7_field_id in dataframes[main_record_set_id].columns:
    # Distribution of GAD-7
    plt.figure(figsize=(8, 4))
    sns.histplot(dataframes[main_record_set_id][gad7_field_id].dropna(), bins=20, kde=True)
    plt.title('Distribution of GAD-7 Total Score')
    plt.xlabel('GAD-7 Score')
    plt.ylabel('Count')
    plt.show()

    if gender_field_id and gender_field_id in dataframes[main_record_set_id].columns:
        plt.figure(figsize=(7, 4))
        sns.boxplot(data=dataframes[main_record_set_id], x=gender_field_id, y=gad7_field_id)
        plt.title('GAD-7 Score Distribution by Gender')
        plt.xlabel('Gender')
        plt.ylabel('GAD-7 Score')
        plt.show()

## 6. Conclusion
This notebook demonstrated how to load a Croissant dataset using the `mlcroissant` library, review available record sets and fields using their `@id`s, and perform basic analysis such as filtering, normalization, grouping, and visualization.

Key findings and next steps:
- The dataset offers rich demographic and mental health survey information, including standardized scores (GAD-7, PHQ-9, PSQ).
- For detailed analysis, refer to specific field and record set `@id`s from the schema.
- You can extend this notebook for advanced analyses and visualizations tailored to your research questions.